# Chapter 18. Combining Datasets: concat and append

## 18.1 Recall: Concatenation of NumPy Arrays

Pandas에서 Series와 DataFrame을 련결하는 것은 NumPy 배렬을 련결하는 것과 류사하게 동작합니다. NumPy에서의 련결은 np.concatenate 함수를 통해 수행됩니다.

In [13]:
import numpy as np
import pandas as pd

x = [1, 2, 3]
y = [4, 5, 6]
z = [7, 8, 9]

arr = np.concatenate([x, y, z])
print(arr)

[1 2 3 4 5 6 7 8 9]


또한 다차원 배렬의 경우 axis 키를 사용하여 련결이 이루어질 축을 지정할 수 있습니다.

In [14]:
x = [[1, 2], [3, 4]]
print(np.concatenate([x, x], axis=1))

[[1 2 1 2]
 [3 4 3 4]]


## 18.2 Simple Concatenation with pd.concat()

pd.concat() 함수는 np.concatenate()와 류사한 문법을 가지며, 몇 가지 추가 option을 제공합니다.

In [15]:
# pd.concat(objs, axis=0, join='outer', ignore_index=False, keys=None,  
#          levels=None, names=None, verify_integrity=False, sort=False, copy=True)

### Basic connection

pd.concat()은 Series나 DataFrame 객체를 간단히 련결하는데 사용할 수 있습니다.

In [16]:
def make_df(cols, ind):
    """Function quickly making DataFrame"""
    data = {c: [str(c) + str(i) for i in ind] for c in cols}
    return pd.DataFrame(data, ind)

# Series connection
ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
print(pd.concat([ser1, ser2]))

1    A
2    B
3    C
4    D
5    E
6    F
dtype: str


In [17]:
# DataFrame connection
df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])
print(pd.concat([df1, df2]))

    A   B
1  A1  B1
2  A2  B2
3  A3  B3
4  A4  B4


### Axis designation

In [ ]:
df3 = make_df('AB', [0, 1])
df4 = make_df('CD', [0, 1])

print(pd.concat([df3, df4], axis='columns'))

    A   B   C   D
0  A0  B0  C0  D0
1  A1  B1  C1  D1


### Duplicate Indices

pd.concat()은 NumPy의 np.concatenate()와 중요한 차이점이 있습니다: Pandas 련결은 결과에 중복 index가 생기더라도 이를 유지합니다.

In [23]:
x = make_df('AB', [0, 1])
y = make_df('AB', [2, 3])
y.index = x.index

print(pd.concat([x, y]))

    A   B
0  A0  B0
1  A1  B1
0  A2  B2
1  A3  B3


### Treating duplicate indexes as errors

In [24]:
try:
    pd.concat([x, y], verify_integrity=True)
except ValueError as e:
    print("ValueError", e)

ValueError Indexes have overlapping values: Index([0, 1], dtype='int64')


### Ignore index

In [25]:
print(pd.concat([x, y], ignore_index=True))

    A   B
0  A0  B0
1  A1  B1
2  A2  B2
3  A3  B3


### Adding MultiIndex Keys

keys option을 사용하면 자료 sourse에 대한 레이블을 지정할 수 있으며, 결과는 계층적으로 indexing된 Series/DataFrame이 됩니다.

In [28]:
print(pd.concat([x, y], keys=['x', 'y']))

      A   B
x 0  A0  B0
  1  A1  B1
y 0  A2  B2
  1  A3  B3


### Connetion and Join

서로 다른 column 집합을 가진 DataFrame을 련결할 때 join 매개변수로 Join 방식을 지정할 수 있습니다.

In [29]:
df5 = make_df('ABC', [1, 2])
df6 = make_df('BCD', [3, 4])

print(pd.concat([df5, df6]))

     A   B   C    D
1   A1  B1  C1  NaN
2   A2  B2  C2  NaN
3  NaN  B3  C3   D3
4  NaN  B4  C4   D4


The default behavior is an outer join of the input columns. You can change it to an inner join by specifying join='inner'.

In [30]:
print(pd.concat([df5, df6], join='inner'))

    B   C
1  B1  C1
2  B2  C2
3  B3  C3
4  B4  C4


### append method

In [32]:
# print(df1.append(df2)) # Error

# Pandas 2.0+ 에서 .append() 제거됨 → pd.concat() 사용